# House Prices: frozen repeated-CV BaseMLP baseline

## tl;dr

- 실제 public 0.12313 제출 recipe를 train-only validation 기준점으로 분리했다.
- 코드 입력은 train.csv 하나이며 test 데이터와 public 점수를 읽지 않는다.
- 15-fold CV는 0.145881 ± 0.026967, repeated OOF는 0.148189, 3-seed ensemble OOF는 0.134232다.
- 기존 repeated-CV 제출의 15개 fold 점수와 세 seed OOF 예측을 정확히 재현했다.
- 이후 feature 후보는 동일 15 folds의 paired delta와 개선 fold 수로 비교한다.
- W&B에는 15개 fold를 한 group으로 묶고 epoch별 loss·RMSLE·learning rate를 기록한다.

## Context & Methods

실제 public 0.12313 제출과 같은 historical BaseMLP recipe를 validation 기준점으로
고정한다. 다만 이 노트북은 순수 validation artifact이므로 test 데이터와 public
점수를 코드에서 읽지 않는다.

### Key Assumptions

- row policy: 모든 학습 행 유지
- modeling copy: `Id=524, YearRemodAdd=2007` 유지
- generated feature: 0개
- validation: `KFold(5, shuffle=True)` × seeds 42, 2026, 3407
- 각 fold에서 전처리기와 BaseMLP를 새로 적합하고 best checkpoint 복원
- 이후 후보는 동일 15개 fold의 paired delta로만 비교
- W&B: fold별 run, experiment ID 기준 group; `WANDB_MODE=disabled`로 비활성화 가능
- 외부 프로젝트 스크립트 import: 0건

## Data

`data/train.csv`만 사용한다. `test.csv`, sample submission, Kaggle public score는
코드 입력에 포함하지 않는다. raw 파일은 수정하지 않는다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import wandb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "validation_test" /
    "basemlp_repeated_cv_baseline.ipynb"
)
REPORT_DIR = ROOT / "reports" / "validation_test"
ARTIFACT_DIR = ROOT / "artifacts" / "validation_test"
EXPERIMENT_ID = "VALBASE-20260720-WANDB-RERUN-NB-04"
RERUN_OF = "VALBASE-20260720-BASEMLP-REPEATED-3SEED-NB-03"
FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_baseline_wandb_fold_results.csv"
OOF_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_baseline_wandb_oof.csv"
SUMMARY_PATH = REPORT_DIR / "basemlp_repeated_cv_baseline_wandb_summary.json"
REFERENCE_FOLD_RESULTS_PATH = (
    REPORT_DIR / "basemlp_repeated_cv_submission_fold_results.csv"
)
REFERENCE_OOF_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_oof.csv"

SPLIT_SEEDS = (42, 2026, 3407)
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "house-prices")
WANDB_ENTITY = os.getenv("WANDB_ENTITY") or None
WANDB_MODE = os.getenv("WANDB_MODE") or None
WANDB_DIR = ARTIFACT_DIR / "wandb"

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
    split_seed: int,
    fold_in_seed: int,
) -> tuple[BaseMLP, dict[str, object]]:
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        group=experiment_id,
        job_type="cv-fold",
        name=f"seed-{split_seed}-fold-{fold_in_seed}",
        dir=str(WANDB_DIR),
        mode=WANDB_MODE,
        force=False,
        config={
            "experiment_id": experiment_id,
            "rerun_of": RERUN_OF,
            "split_seed": split_seed,
            "fold": fold_in_seed,
            "split_number": fold,
            "model_seed": seed,
            "n_splits": N_SPLITS,
            "training_rows": len(X_train),
            "validation_rows": len(X_valid),
            "input_dim": X_train.shape[1],
            "hidden_dims": list(HIDDEN_DIMS),
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "target": "fold-standardized log1p(SalePrice)",
        },
    )
    wandb_run.define_metric("epoch")
    wandb_run.define_metric("train/loss", step_metric="epoch", summary="min")
    wandb_run.define_metric("validation/loss", step_metric="epoch", summary="min")
    wandb_run.define_metric("validation/rmsle", step_metric="epoch", summary="min")
    wandb_run.define_metric("optimizer/learning_rate", step_metric="epoch", summary="last")
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_row = {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        epoch_rows.append(epoch_row)
        wandb_run.log({
            "epoch": epoch,
            "train/loss": epoch_row["train_loss"],
            "validation/loss": epoch_row["validation_loss"],
            "validation/rmsle": valid_rmsle,
            "optimizer/learning_rate": epoch_row["learning_rate"],
        })

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    wandb_run.summary.update({
        "best/epoch": best_epoch,
        "best/validation_rmsle": best_score,
        "restored/validation_rmsle": restored_rmsle,
        "stopping_epoch": epoch,
        "checkpoint_path": str(checkpoint_path.relative_to(ROOT)),
        "epoch_log_path": str(epoch_log_path.relative_to(ROOT)),
    })
    wandb_run_id = wandb_run.id
    wandb_run_url = wandb_run.url
    wandb_run_dir = wandb_run.dir
    wandb_run_mode = str(wandb_run.settings.mode)
    wandb_run.finish()
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
        "wandb_run_id": wandb_run_id,
        "wandb_run_url": wandb_run_url or "",
        "wandb_run_dir": wandb_run_dir,
        "wandb_run_mode": wandb_run_mode,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(
    TRAIN_PATH, keep_default_na=False
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert train.shape == (1460, 81)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("data scope: train.csv only")
print("external project script imports: 0")

train: 1,460 rows × 81 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
data scope: train.csv only
external project script imports: 0


## Fixed preprocessing and model input

기존 BaseMLP의 구조적 결측 처리와 fold-fitted numeric/categorical pipeline을
유지한다. `Id`는 모델 feature에서 제외된다.

In [2]:
def clean_rows_historical_basemlp(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned

X = clean_rows_historical_basemlp(
    train.drop(columns="SalePrice"), categorical_columns
)
assert float(
    X.loc[X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2007
assert X.shape == (1460, 80)
print("Historical keep-all BaseMLP input fixed; generated features: 0")


def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

Historical keep-all BaseMLP input fixed; generated features: 0


## Repeated 15-fold training

각 fold에서 전처리기, 모델, optimizer를 새로 만들고 validation RMSLE 기준
best checkpoint를 복원한다. 기존 repeated-CV 제출의 fold 점수와 대조한다.

In [3]:
OUTPUT_DIR = ARTIFACT_DIR / EXPERIMENT_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
PREPROCESSOR_DIR = OUTPUT_DIR / "preprocessors"
EPOCH_LOG_DIR = OUTPUT_DIR / "logs"
for directory in (
    CHECKPOINT_DIR, PREPROCESSOR_DIR, EPOCH_LOG_DIR, WANDB_DIR
):
    directory.mkdir(parents=True, exist_ok=True)

oof_log_by_seed = np.full(
    (len(SPLIT_SEEDS), len(train)), np.nan, dtype=np.float64
)
fold_rows: list[dict[str, object]] = []
started = time.perf_counter()

for seed_index, split_seed in enumerate(SPLIT_SEEDS):
    splitter = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=split_seed,
    )
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(X), start=1
    ):
        split_number = seed_index * N_SPLITS + fold
        model_seed = split_seed + fold
        checkpoint_path = (
            CHECKPOINT_DIR /
            f"seed_{split_seed}_fold_{fold}_best.pt"
        )
        preprocessor_path = (
            PREPROCESSOR_DIR /
            f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
        )
        epoch_log_path = (
            EPOCH_LOG_DIR /
            f"seed_{split_seed}_fold_{fold}_epochs.csv"
        )

        preprocessor = make_preprocessor(list(NUMERIC_COLUMNS))
        X_train = preprocessor.fit_transform(
            X.iloc[train_index]
        ).astype(np.float32)
        X_valid = preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        joblib.dump(preprocessor, preprocessor_path)

        model, training_result = train_fold(
            X_train,
            y_log[train_index],
            X_valid,
            y_log[valid_index],
            seed=model_seed,
            experiment_id=EXPERIMENT_ID,
            fold=split_number,
            checkpoint_path=checkpoint_path,
            epoch_log_path=epoch_log_path,
            split_seed=split_seed,
            fold_in_seed=fold,
        )
        valid_log_prediction = predict_log_target(
            model,
            X_valid,
            training_result["target_mean"],
            training_result["target_std"],
        )
        oof_log_by_seed[seed_index, valid_index] = (
            valid_log_prediction
        )
        fold_rows.append({
            "split_seed": split_seed,
            "fold": fold,
            "split_number": split_number,
            "model_seed": model_seed,
            "training_rows": len(train_index),
            "validation_rows": len(valid_index),
            "encoded_features": X_train.shape[1],
            "best_epoch": training_result["best_epoch"],
            "stopping_epoch": training_result["stopping_epoch"],
            "rmsle": training_result[
                "restored_validation_rmsle"
            ],
            "checkpoint_path": str(
                checkpoint_path.relative_to(ROOT)
            ),
            "preprocessor_path": str(
                preprocessor_path.relative_to(ROOT)
            ),
            "epoch_log_path": str(
                epoch_log_path.relative_to(ROOT)
            ),
            "wandb_run_id": training_result["wandb_run_id"],
            "wandb_run_url": training_result["wandb_run_url"],
            "wandb_run_dir": training_result["wandb_run_dir"],
            "wandb_run_mode": training_result["wandb_run_mode"],
        })

assert not np.isnan(oof_log_by_seed).any()
fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(FOLD_RESULTS_PATH, index=False)
duration_seconds = time.perf_counter() - started

reference_folds = pd.read_csv(REFERENCE_FOLD_RESULTS_PATH).sort_values(
    ["split_seed", "fold"]
)
observed_folds = fold_results.sort_values(
    ["split_seed", "fold"]
)
assert reference_folds[["split_seed", "fold"]].to_numpy().tolist() == (
    observed_folds[["split_seed", "fold"]].to_numpy().tolist()
)
assert np.allclose(
    reference_folds["rmsle"].to_numpy(),
    observed_folds["rmsle"].to_numpy(),
    rtol=0,
    atol=1e-12,
)
print(f"15 folds completed in {duration_seconds:.2f} seconds")
print("All fold scores exactly reproduce the repeated-CV submission recipe.")

wandb: Tracking run with wandb version 0.28.1


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /Users/joonha/workspace/House_Prices/artifacts/validation_test/wandb/wandb/offline-run-20260720_115629-1qmqvuwv
wandb: View this run in the terminal with `wandb leet`


wandb: WARNING URL not available in offline run


wandb: 
wandb: Run history:
wandb:                   epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb: optimizer/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss █▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         validation/loss █▅▄▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂
wandb:        validation/rmsle █▅▄▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▂▂▂▂
wandb: 
wandb: Run summary:
wandb:                best/epoch 12
wandb:     best/validation_rmsle 0.14089
wandb:           checkpoint_path artifacts/validation...
wandb:                     epoch 37
wandb:            epoch_log_path artifacts/validation...
wandb: restored/validation_rmsle 0.14089
wandb:            stopping_epoch 37
wandb: 


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /Users/joonha/workspace/House_Prices/artifacts/validation_test/wandb/wandb/offline-run-20260720_115629-1qmqvuwv


...

wandb: WARNING URL not available in offline run


wandb: 
wandb: Run history:
wandb:                   epoch ▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
wandb: optimizer/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss █▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         validation/loss █▆▃▂▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▃▂▃▃▃▃▂▃
wandb:        validation/rmsle █▆▃▂▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▃▃▂▂▂▃▃▃▃▃▃▃▃▃
wandb: 
wandb: Run summary:
wandb:                best/epoch 8
wandb:     best/validation_rmsle 0.10909
wandb:           checkpoint_path artifacts/validation...
wandb:                     epoch 33
wandb:            epoch_log_path artifacts/validation...
wandb: restored/validation_rmsle 0.10909
wandb:            stopping_epoch 33
wandb: 


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /Users/joonha/workspace/House_Prices/artifacts/validation_test/wandb/wandb/offline-run-20260720_115647-p8r0ekrx


wandb: Find logs at: /Users/joonha/workspace/House_Prices/artifacts/validation_test/wandb/wandb/offline-run-20260720_115647-p8r0ekrx/logs


15 folds completed in 18.82 seconds
All fold scores exactly reproduce the repeated-CV submission recipe.


## Results and frozen comparison contract

15-fold 평균을 주 지표로, 반복 OOF·3-seed ensemble OOF·paired 개선 fold 수를
보조 지표로 사용한다. 절대 CV delta 0.0005 미만은 동률로 취급한다.

In [4]:
fold_scores = fold_results["rmsle"].to_numpy(
    dtype=np.float64
)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
seed_oof_scores = {
    str(split_seed): float(np.sqrt(np.mean(
        (y_log - oof_log_by_seed[seed_index]) ** 2
    )))
    for seed_index, split_seed in enumerate(SPLIT_SEEDS)
}
repeated_oof_rmsle = float(np.sqrt(np.mean(
    (
        np.tile(y_log, (len(SPLIT_SEEDS), 1))
        - oof_log_by_seed
    ) ** 2
)))
ensemble_oof_log_prediction = oof_log_by_seed.mean(axis=0)
three_seed_ensemble_oof_rmsle = float(np.sqrt(np.mean(
    (y_log - ensemble_oof_log_prediction) ** 2
)))

oof_results = pd.DataFrame({
    "Id": train["Id"].to_numpy(dtype=np.int64),
    "actual_log1p": y_log,
})
for seed_index, split_seed in enumerate(SPLIT_SEEDS):
    oof_results[f"seed_{split_seed}_oof_log_prediction"] = (
        oof_log_by_seed[seed_index]
    )
oof_results["three_seed_ensemble_oof_log_prediction"] = (
    ensemble_oof_log_prediction
)
oof_results["three_seed_ensemble_squared_log_error"] = (
    y_log - ensemble_oof_log_prediction
) ** 2
oof_results.to_csv(OOF_RESULTS_PATH, index=False)

reference_oof = pd.read_csv(REFERENCE_OOF_PATH)
assert reference_oof["Id"].equals(oof_results["Id"])
for split_seed in SPLIT_SEEDS:
    assert np.allclose(
        reference_oof[
            f"seed_{split_seed}_oof_log_prediction"
        ].to_numpy(),
        oof_results[
            f"seed_{split_seed}_oof_log_prediction"
        ].to_numpy(),
        rtol=0,
        atol=1e-12,
    )

artifact_checks = {
    "checkpoint_count": len(list(CHECKPOINT_DIR.glob("*.pt"))),
    "preprocessor_count": len(list(PREPROCESSOR_DIR.glob("*.joblib"))),
    "epoch_log_count": len(list(EPOCH_LOG_DIR.glob("*.csv"))),
}
assert artifact_checks == {
    "checkpoint_count": 15,
    "preprocessor_count": 15,
    "epoch_log_count": 15,
}
assert np.isclose(cv_mean, 0.14588108497354113, rtol=0, atol=1e-12)
assert np.isclose(cv_std, 0.026966873871957453, rtol=0, atol=1e-12)
assert np.isclose(
    three_seed_ensemble_oof_rmsle,
    0.1342324057183868,
    rtol=0,
    atol=1e-12,
)

run_timestamp = datetime.now(timezone.utc).isoformat()
summary = {
    "run_timestamp_utc": run_timestamp,
    "experiment_id": EXPERIMENT_ID,
    "rerun_of": RERUN_OF,
    "role": "frozen primary validation baseline for future paired feature experiments",
    "data_scope": {
        "train_path": "data/train.csv",
        "train_sha256": sha256(TRAIN_PATH),
        "test_data_loaded": False,
        "public_scores_loaded": False,
        "all_training_rows_retained": True,
    },
    "recipe": {
        "splitter": "KFold(5, shuffle=True)",
        "split_seeds": list(SPLIT_SEEDS),
        "model_seed": "split_seed + fold",
        "preprocessing": "fold median + StandardScaler + one-hot; structural absence labels",
        "id_524_yearremodadd": 2007,
        "generated_features": [],
        "model": "BaseMLP input->128(ReLU)->64(ReLU)->1",
        "optimizer": "Adam(lr=0.001, weight_decay=0)",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
    },
    "results": {
        "fold_scores": fold_scores.tolist(),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "seed_oof_rmsle": seed_oof_scores,
        "repeated_oof_rmsle": repeated_oof_rmsle,
        "three_seed_ensemble_oof_rmsle": three_seed_ensemble_oof_rmsle,
        "duration_seconds": duration_seconds,
    },
    "comparison_contract": {
        "primary_metric": "mean RMSLE across the same 15 folds",
        "comparison": "paired candidate-minus-baseline delta for each identical seed/fold",
        "secondary_metrics": [
            "repeated OOF RMSLE",
            "three-seed ensemble OOF RMSLE",
            "count of improved paired folds",
        ],
        "small_delta_rule": "treat absolute CV delta below 0.0005 as tied",
        "public_leaderboard_for_selection": False,
    },
    "validation": {
        "source_fold_scores_exactly_reproduced": True,
        "source_seed_oof_predictions_exactly_reproduced": True,
        **artifact_checks,
        "external_project_script_imports": 0,
    },
    "tracking": {
        "wandb_project": WANDB_PROJECT,
        "wandb_entity": WANDB_ENTITY,
        "wandb_requested_mode": WANDB_MODE or "auto",
        "wandb_observed_modes": sorted(
            fold_results["wandb_run_mode"].unique().tolist()
        ),
        "wandb_group": EXPERIMENT_ID,
        "wandb_run_count": len(fold_results),
        "wandb_run_ids": fold_results["wandb_run_id"].tolist(),
        "wandb_run_urls": [
            url for url in fold_results["wandb_run_url"].tolist() if url
        ],
    },
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "fold_results": str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        "oof_predictions": str(OOF_RESULTS_PATH.relative_to(ROOT)),
        "checkpoint_dir": str(CHECKPOINT_DIR.relative_to(ROOT)),
        "preprocessor_dir": str(PREPROCESSOR_DIR.relative_to(ROOT)),
        "epoch_log_dir": str(EPOCH_LOG_DIR.relative_to(ROOT)),
        "wandb_dir": str(WANDB_DIR.relative_to(ROOT)),
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

append_experiment({
    "experiment_id": EXPERIMENT_ID,
    "datetime": run_timestamp,
    "objective": "Add W&B fold/epoch tracking and rerun the frozen repeated-CV BaseMLP baseline",
    "data_version": f"train sha256={sha256(TRAIN_PATH)}",
    "split_strategy": "KFold(5,shuffle=True) repeated at seeds 42|2026|3407",
    "folds": "5x3",
    "seed": "42|2026|3407; model seed=split_seed+fold",
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "Historical BaseMLP fold preprocessing; Id=524 YearRemodAdd=2007 modeling-copy correction; train only",
    "features": "79 original predictors; Id excluded; generated features 0",
    "model": "Manual PyTorch BaseMLP validation baseline",
    "architecture": "input->128(ReLU)->64(ReLU)->1; Kaiming hidden/Xavier output",
    "optimizer": "Adam(weight_decay=0)",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "best_epoch": json.dumps(
        fold_results["best_epoch"].astype(int).tolist()
    ),
    "hyperparameters": f"rerun_of={RERUN_OF}; test/public not loaded; keep_all; split seeds fixed; W&B project={WANDB_PROJECT}; group={EXPERIMENT_ID}; one run per fold",
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "holdout_score": three_seed_ensemble_oof_rmsle,
    "checkpoint_path": str(CHECKPOINT_DIR.relative_to(ROOT)),
    "artifact_path": " | ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        str(OOF_RESULTS_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
    ]),
    "result": "wandb_instrumented_validation_baseline_ready",
    "interpretation": "W&B tracking was added without changing the model recipe; the rerun exactly reproduces the reference fold scores and seed-level OOF predictions.",
    "next_action": "Sync offline W&B runs or set WANDB_MODE=online after login; run feature candidates on the identical 15 folds",
})

display(pd.DataFrame({
    "metric": [
        "15-fold CV mean",
        "15-fold CV std",
        "repeated OOF RMSLE",
        "3-seed ensemble OOF RMSLE",
    ],
    "value": [
        cv_mean,
        cv_std,
        repeated_oof_rmsle,
        three_seed_ensemble_oof_rmsle,
    ],
}))
display(fold_results)
print(f"Summary: {SUMMARY_PATH.relative_to(ROOT)}")

,metric,value
0,15-fold CV mean,0.145881
1,15-fold CV std,0.026967
2,repeated OOF RMSLE,0.148189
3,3-seed ensemble OOF RMSLE,0.134232


,split_seed,fold,split_number,model_seed,training_rows,validation_rows,encoded_features,best_epoch,stopping_epoch,rmsle,checkpoint_path,preprocessor_path,epoch_log_path,wandb_run_id,wandb_run_url,wandb_run_dir,wandb_run_mode
0,42,1,1,43,1168,292,327,12,37,0.140891,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,1qmqvuwv,,/Users/joonha/workspace/House_Prices/artifacts...,offline
1,42,2,2,44,1168,292,327,43,68,0.131788,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,xcbvqi8v,,/Users/joonha/workspace/House_Prices/artifacts...,offline
2,42,3,3,45,1168,292,323,5,30,0.222199,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,kvbtt8k5,,/Users/joonha/workspace/House_Prices/artifacts...,offline
3,42,4,4,46,1168,292,324,21,46,0.134316,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,nurvirrg,,/Users/joonha/workspace/House_Prices/artifacts...,offline
4,42,5,5,47,1168,292,328,14,39,0.122514,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,5ulxpdla,,/Users/joonha/workspace/House_Prices/artifacts...,offline
5,2026,1,6,2027,1168,292,323,7,32,0.141128,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,fynakbz1,,/Users/joonha/workspace/House_Prices/artifacts...,offline
6,2026,2,7,2028,1168,292,326,30,55,0.179038,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,swb7t925,,/Users/joonha/workspace/House_Prices/artifacts...,offline
7,2026,3,8,2029,1168,292,325,5,30,0.128823,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,iw0a0u5e,,/Users/joonha/workspace/House_Prices/artifacts...,offline
8,2026,4,9,2030,1168,292,327,11,36,0.149235,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,3kt5fywf,,/Users/joonha/workspace/House_Prices/artifacts...,offline
9,2026,5,10,2031,1168,292,328,11,36,0.141151,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,artifacts/validation_test/VALBASE-20260720-WAN...,iooyvlc1,,/Users/joonha/workspace/House_Prices/artifacts...,offline


Summary: reports/validation_test/basemlp_repeated_cv_baseline_wandb_summary.json


## Takeaways

- primary baseline: 15-fold mean RMSLE 0.145881 ± 0.026967
- secondary baseline: repeated OOF 0.148189, 3-seed ensemble OOF 0.134232
- candidate 비교는 seed와 fold가 동일한 paired delta를 사용하고, 절대 CV delta 0.0005 미만은 동률로 취급한다.
- 반복 OOF·ensemble OOF가 같은 방향이고 개선 fold가 충분한 후보만 유지한다.
- test/public 정보는 이 baseline 생성과 향후 feature 선택에 사용하지 않는다.
- public 0.12313은 이 recipe의 이미 관측된 제출 결과일 뿐 validation 입력은 아니다.
- W&B 15개 fold run은 experiment ID group으로 묶이며, offline 실행은 로그인 후 `wandb sync`할 수 있다.